# Perbaikan ekspor backend RacikAI
Bagian 18 mengekspor seluruh subfolder, model, index, metadata, checksum, dan memeriksa kecocokan embedding. Jika hasil training masih tersedia, jalankan pemuatan model dan bagian ekspor; tidak perlu training ulang. Notebook asli tidak diubah.


# 🍳 Recipe RAG — Kaggle End-to-End Notebook


Notebook ini dirancang untuk **Kaggle Notebook** menggunakan:

- **Dataset:** Food Ingredients and Recipes Dataset with Images
- **Embedding model:** `intfloat/multilingual-e5-base`
- **Fine-tuning:** Sentence Transformers
- **Vector search:** FAISS
- **LLM:** Gemini API via Kaggle Secrets
- **Query:** Bahasa Indonesia
- **Corpus:** resep berbahasa Inggris dari dataset

> Kita tidak melatih LLM dari nol. Yang di-*fine-tune* adalah model embedding/retrieval agar pencarian resep lebih sesuai dengan domain.

## Sebelum menjalankan

1. Di Kaggle, klik **Add Input** lalu tambahkan dataset **Food Ingredients and Recipes Dataset with Images**.
2. Aktifkan **GPU** melalui Notebook Settings bila tersedia.
3. Aktifkan **Internet** agar notebook dapat mengunduh model/package dan mengakses Gemini API.
4. Untuk tahap LLM, tambahkan Kaggle Secret bernama `GEMINI_API_KEY`.

Dataset:
https://www.kaggle.com/datasets/pes12017000148/food-ingredients-and-recipe-dataset-with-images

Embedding model:
https://huggingface.co/intfloat/multilingual-e5-base

## 0. Install Library dan Konfigurasi

`FAST_MODE=True` cocok untuk percobaan awal. Untuk eksperimen final, ubah menjadi `False`.

In [ ]:
!pip install -q -U sentence-transformers faiss-cpu datasets google-genai accelerate

In [ ]:
import os
import re
import ast
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import faiss

from sklearn.model_selection import train_test_split
from datasets import Dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FAST_MODE = True
MAX_TRAIN_SAMPLES = 4000
MAX_EVAL_QUERIES = 300

EPOCHS = 1
TRAIN_BATCH_SIZE = 8
ENCODE_BATCH_SIZE = 64
TOP_K = 5

BASE_MODEL_NAME = "intfloat/multilingual-e5-base"
MODEL_DIR = "/kaggle/working/recipe-e5-finetuned"
FAISS_PATH = "/kaggle/working/recipe_faiss.index"
METADATA_PATH = "/kaggle/working/recipe_metadata.csv"

print("Device:", DEVICE)
print("FAST_MODE:", FAST_MODE)

# 1. Dataset

Notebook akan mencari CSV resep secara otomatis dari `/kaggle/input`.

In [ ]:
input_root = Path("/kaggle/input")
csv_files = list(input_root.rglob("*.csv"))

print(f"Jumlah CSV ditemukan: {len(csv_files)}")
for p in csv_files[:20]:
    print("-", p)

In [ ]:
REQUIRED_COLUMNS = {"Title", "Ingredients", "Instructions"}

dataset_path = None

for path in csv_files:
    try:
        sample = pd.read_csv(path, nrows=5)
        if REQUIRED_COLUMNS.issubset(sample.columns):
            dataset_path = path
            break
    except Exception:
        pass

if dataset_path is None:
    raise FileNotFoundError(
        "Dataset resep belum ditemukan. Tambahkan dataset melalui Add Input di Kaggle."
    )

print("Dataset:", dataset_path)

df_raw = pd.read_csv(dataset_path)
print("Shape:", df_raw.shape)

display(df_raw.head())

# 2. EDA — Exploratory Data Analysis

Kita cek ukuran data, tipe kolom, missing values, duplicate, distribusi panjang teks, dan ingredients paling sering.

In [ ]:
print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

print("\nData types:")
display(df_raw.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    df_raw.isna()
          .sum()
          .sort_values(ascending=False)
          .to_frame("missing")
)

print("\nDuplicate rows:", df_raw.duplicated().sum())
print("Duplicate Title:", df_raw.duplicated(subset=["Title"]).sum())

In [ ]:
eda_df = df_raw.copy()

eda_df["ingredients_chars"] = (
    eda_df["Ingredients"].fillna("").astype(str).str.len()
)

eda_df["instructions_chars"] = (
    eda_df["Instructions"].fillna("").astype(str).str.len()
)

display(
    eda_df[["ingredients_chars", "instructions_chars"]]
    .describe()
    .round(2)
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(eda_df["instructions_chars"], bins=50)
plt.title("Distribusi Panjang Instructions")
plt.xlabel("Jumlah karakter")
plt.ylabel("Jumlah resep")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(eda_df["ingredients_chars"], bins=50)
plt.title("Distribusi Panjang Ingredients")
plt.xlabel("Jumlah karakter")
plt.ylabel("Jumlah resep")
plt.show()

In [ ]:
def parse_list_like(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]

    text = str(value).strip()

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple)):
            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]
    except Exception:
        pass

    return [
        x.strip()
        for x in re.split(r",|\n|;", text)
        if x.strip()
    ]


ingredient_source = (
    "Cleaned_Ingredients"
    if "Cleaned_Ingredients" in df_raw.columns
    else "Ingredients"
)

all_ingredients = []

for value in df_raw[ingredient_source].dropna():
    all_ingredients.extend(parse_list_like(value))

top_ingredients = Counter(
    x.lower() for x in all_ingredients
).most_common(20)

top_ingredients_df = pd.DataFrame(
    top_ingredients,
    columns=["ingredient", "count"]
)

display(top_ingredients_df)

In [ ]:
plot_df = top_ingredients_df.sort_values("count", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["ingredient"], plot_df["count"])
plt.title("20 Ingredients Paling Sering Muncul")
plt.xlabel("Frekuensi")
plt.ylabel("Ingredient")
plt.show()

# 3. Preprocessing & Cleaning

Tahap ini:
- menghapus data penting yang kosong,
- menghapus judul duplikat,
- merapikan whitespace,
- parsing ingredients,
- membuat `recipe_id`.

In [ ]:
def normalize_whitespace(text):
    text = "" if pd.isna(text) else str(text)
    return re.sub(r"\s+", " ", text).strip()


df = df_raw.copy()

df["Title"] = df["Title"].apply(normalize_whitespace)
df["Ingredients"] = df["Ingredients"].apply(normalize_whitespace)
df["Instructions"] = df["Instructions"].apply(normalize_whitespace)

if "Cleaned_Ingredients" not in df.columns:
    df["Cleaned_Ingredients"] = df["Ingredients"]

df = df[
    (df["Title"] != "") &
    (df["Ingredients"] != "") &
    (df["Instructions"] != "")
].copy()

df["title_key"] = (
    df["Title"]
    .str.lower()
    .str.strip()
)

df = (
    df.drop_duplicates(subset=["title_key"])
      .reset_index(drop=True)
)

df["ingredient_list"] = (
    df["Cleaned_Ingredients"]
    .apply(parse_list_like)
)

empty_mask = df["ingredient_list"].str.len() == 0

df.loc[empty_mask, "ingredient_list"] = (
    df.loc[empty_mask, "Ingredients"]
      .apply(parse_list_like)
)

df["ingredient_text"] = df["ingredient_list"].apply(
    lambda xs: ", ".join(xs)
)

df["recipe_id"] = np.arange(len(df), dtype=int)

print("Shape setelah cleaning:", df.shape)

display(
    df[
        ["recipe_id", "Title", "ingredient_text", "Instructions"]
    ].head()
)

In [ ]:
cleaning_summary = pd.DataFrame({
    "metric": [
        "raw_rows",
        "clean_rows",
        "rows_removed",
        "missing_title_after_clean",
        "missing_ingredients_after_clean",
        "missing_instructions_after_clean",
    ],
    "value": [
        len(df_raw),
        len(df),
        len(df_raw) - len(df),
        (df["Title"] == "").sum(),
        (df["Ingredients"] == "").sum(),
        (df["Instructions"] == "").sum(),
    ]
})

display(cleaning_summary)

# 4. Document RAG

Kita membuat:

- `rag_document`: dokumen lengkap untuk LLM.
- `retrieval_document`: teks yang di-embedding untuk retrieval.
- `passage_text`: retrieval document dengan prefix `passage:`.

Untuk E5:
- query diberi prefix `query:`
- dokumen diberi prefix `passage:`

In [ ]:
def build_rag_document(row):
    return (
        f"Title: {row['Title']}\n"
        f"Ingredients: {row['ingredient_text']}\n"
        f"Instructions: {row['Instructions']}"
    )


df["rag_document"] = df.apply(
    build_rag_document,
    axis=1
)

df["retrieval_document"] = df.apply(
    lambda row: (
        f"Title: {row['Title']}. "
        f"Ingredients: {row['ingredient_text']}. "
        f"Instructions: {row['Instructions']}"
    ),
    axis=1,
)

df["passage_text"] = (
    "passage: " + df["retrieval_document"]
)

print(df.loc[0, "rag_document"][:1500])

In [ ]:
df["rag_chars"] = df["rag_document"].str.len()

display(
    df["rag_chars"]
    .describe()
    .to_frame("rag_document_chars")
)

plt.figure(figsize=(8, 4))
plt.hist(df["rag_chars"], bins=50)
plt.title("Distribusi Panjang RAG Document")
plt.xlabel("Jumlah karakter")
plt.ylabel("Jumlah resep")
plt.show()

# 5. Membuat Data Training

Dataset asli belum memiliki pasangan `query → resep relevan`. Karena itu kita membuat **synthetic queries / weak supervision**.

Positive document = resep asal query.

Negative document akan dibuat setelah split agar data test tidak bocor ke training.

In [ ]:
QUERY_TEMPLATES = [
    "resep {title}",
    "cara membuat {title}",
    "saya ingin memasak {title}",
    "saya punya {ingredients}. bisa masak apa?",
    "makanan apa yang bisa dibuat dari {ingredients}?",
    "rekomendasikan resep dengan bahan {ingredients}",
]


def short_ingredients(items, n=4):
    items = [
        normalize_whitespace(x)
        for x in items
        if normalize_whitespace(x)
    ]

    if not items:
        return "bahan yang tersedia"

    return ", ".join(items[:n])


def make_query(row):
    rng = random.Random(
        SEED + int(row["recipe_id"])
    )

    template = rng.choice(QUERY_TEMPLATES)

    query = template.format(
        title=row["Title"],
        ingredients=short_ingredients(
            row["ingredient_list"],
            n=4
        )
    )

    return "query: " + query


pair_df = df[
    [
        "recipe_id",
        "Title",
        "ingredient_list",
        "passage_text"
    ]
].copy()

pair_df["anchor"] = df.apply(
    make_query,
    axis=1
)

pair_df["positive"] = pair_df["passage_text"]

display(
    pair_df[
        ["recipe_id", "anchor", "positive"]
    ].head(10)
)

# 6. Splitting Train / Validation / Test

- 80% train
- 10% validation
- 10% test

Setelah split, negative recipe dipilih dari split yang sama dengan overlap ingredient rendah.

In [ ]:
train_pairs, temp_pairs = train_test_split(
    pair_df,
    test_size=0.20,
    random_state=SEED,
)

val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=0.50,
    random_state=SEED,
)

train_pairs = train_pairs.reset_index(drop=True)
val_pairs = val_pairs.reset_index(drop=True)
test_pairs = test_pairs.reset_index(drop=True)

print("Train:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Test:", len(test_pairs))

In [ ]:
def ingredient_set(items):
    return {
        re.sub(
            r"[^a-z0-9 ]",
            "",
            str(x).lower()
        ).strip()
        for x in items
        if str(x).strip()
    }


def add_negatives(split_df, candidate_trials=25):
    split_df = (
        split_df.copy()
        .reset_index(drop=True)
    )

    rng = np.random.default_rng(SEED)

    ingredient_sets = [
        ingredient_set(x)
        for x in split_df["ingredient_list"]
    ]

    positive_texts = (
        split_df["positive"]
        .tolist()
    )

    negative_texts = []

    for i, current_set in enumerate(ingredient_sets):
        candidates = rng.integers(
            0,
            len(split_df),
            size=candidate_trials
        )

        candidates = [
            int(c)
            for c in candidates
            if int(c) != i
        ]

        if not candidates:
            candidates = [
                (i + 1) % len(split_df)
            ]

        best_idx = candidates[0]
        best_overlap = float("inf")

        for candidate_idx in candidates:
            other_set = ingredient_sets[candidate_idx]
            union = current_set | other_set

            overlap = (
                len(current_set & other_set) / len(union)
                if union
                else 0.0
            )

            if overlap < best_overlap:
                best_overlap = overlap
                best_idx = candidate_idx

        negative_texts.append(
            positive_texts[best_idx]
        )

    split_df["negative"] = negative_texts

    return split_df


train_pairs = add_negatives(train_pairs)
val_pairs = add_negatives(val_pairs)
test_pairs = add_negatives(test_pairs)

display(
    train_pairs[
        ["anchor", "positive", "negative"]
    ].head(3)
)

In [ ]:
if FAST_MODE:
    train_used = train_pairs.sample(
        n=min(
            MAX_TRAIN_SAMPLES,
            len(train_pairs)
        ),
        random_state=SEED,
    ).reset_index(drop=True)
else:
    train_used = train_pairs.copy()


eval_used = test_pairs.sample(
    n=min(
        MAX_EVAL_QUERIES,
        len(test_pairs)
    ),
    random_state=SEED,
).reset_index(drop=True)

print("Training samples:", len(train_used))
print("Evaluation queries:", len(eval_used))

# 7. Model Awal

Kita memakai `intfloat/multilingual-e5-base`.

Sebelum fine-tuning, model dievaluasi terlebih dahulu sebagai baseline.

In [ ]:
base_model = SentenceTransformer(
    BASE_MODEL_NAME,
    device=DEVICE,
)

base_model.max_seq_length = 384

print("Model:", BASE_MODEL_NAME)
print(
    "Embedding dimension:",
    base_model.get_sentence_embedding_dimension()
)
print(
    "Max sequence length:",
    base_model.max_seq_length
)

## Evaluasi Retrieval

Metrik:
- Recall@1
- Recall@5
- Recall@10
- MRR@10

In [ ]:
def evaluate_retrieval(
    model,
    queries_df,
    corpus_df,
    max_k=10,
    batch_size=64,
):
    corpus_texts = (
        corpus_df["passage_text"]
        .tolist()
    )

    corpus_embeddings = model.encode(
        corpus_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    index = faiss.IndexFlatIP(
        corpus_embeddings.shape[1]
    )

    index.add(corpus_embeddings)

    query_embeddings = model.encode(
        queries_df["anchor"].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = index.search(
        query_embeddings,
        max_k
    )

    corpus_recipe_ids = (
        corpus_df["recipe_id"]
        .to_numpy()
    )

    relevant_ids = (
        queries_df["recipe_id"]
        .to_numpy()
    )

    hit_1 = 0
    hit_5 = 0
    hit_10 = 0
    reciprocal_ranks = []

    for row_idx, relevant_id in enumerate(relevant_ids):
        retrieved_ids = corpus_recipe_ids[
            indices[row_idx]
        ]

        positions = np.where(
            retrieved_ids == relevant_id
        )[0]

        if len(positions) > 0:
            rank = int(positions[0]) + 1
        else:
            rank = None

        hit_1 += int(
            rank is not None and rank <= 1
        )

        hit_5 += int(
            rank is not None and rank <= 5
        )

        hit_10 += int(
            rank is not None and rank <= 10
        )

        reciprocal_ranks.append(
            1.0 / rank
            if rank is not None and rank <= 10
            else 0.0
        )

    n = len(queries_df)

    return {
        "Recall@1": hit_1 / n,
        "Recall@5": hit_5 / n,
        "Recall@10": hit_10 / n,
        "MRR@10": float(
            np.mean(reciprocal_ranks)
        ),
    }

In [ ]:
base_metrics = evaluate_retrieval(
    model=base_model,
    queries_df=eval_used,
    corpus_df=df,
    max_k=10,
    batch_size=ENCODE_BATCH_SIZE,
)

print("BASE MODEL METRICS")

for metric, value in base_metrics.items():
    print(
        f"{metric}: {value:.4f}"
    )

# 8. Fine-Tuning

Format:

```text
anchor   = query
positive = resep relevan
negative = resep tidak relevan
```

Loss: `MultipleNegativesRankingLoss`.

In [ ]:
train_dataset = Dataset.from_dict({
    "anchor": train_used["anchor"].tolist(),
    "positive": train_used["positive"].tolist(),
    "negative": train_used["negative"].tolist(),
})

val_dataset = Dataset.from_dict({
    "anchor": val_pairs["anchor"].tolist(),
    "positive": val_pairs["positive"].tolist(),
    "negative": val_pairs["negative"].tolist(),
})

print(train_dataset)
print(val_dataset)

In [ ]:
train_loss = (
    losses.MultipleNegativesRankingLoss(
        base_model
    )
)

training_args = SentenceTransformerTrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    bf16=False,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = SentenceTransformerTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss,
)

trainer.train()

In [ ]:
base_model.save_pretrained(
    MODEL_DIR
)

finetuned_model = SentenceTransformer(
    MODEL_DIR,
    device=DEVICE,
)

finetuned_model.max_seq_length = 384

print(
    "Fine-tuned model:",
    MODEL_DIR
)

# 9. Evaluasi Setelah Fine-Tuning

Bandingkan base model dan model yang sudah di-fine-tune menggunakan test query yang sama.

In [ ]:
finetuned_metrics = evaluate_retrieval(
    model=finetuned_model,
    queries_df=eval_used,
    corpus_df=df,
    max_k=10,
    batch_size=ENCODE_BATCH_SIZE,
)

comparison = pd.DataFrame(
    [
        base_metrics,
        finetuned_metrics
    ],
    index=[
        "Base multilingual-e5-base",
        "Fine-tuned Recipe E5"
    ]
).T

display(
    comparison.round(4)
)

In [ ]:
comparison.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title(
    "Base vs Fine-Tuned Retrieval"
)

plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

In [ ]:
print("PERUBAHAN SETELAH FINE-TUNING")

for metric in base_metrics:
    diff = (
        finetuned_metrics[metric]
        - base_metrics[metric]
    )

    sign = "+" if diff >= 0 else ""

    print(
        f"{metric}: {sign}{diff:.4f}"
    )

# 10. Embedding Seluruh Dataset → FAISS

Setelah model final tersedia:
1. encode seluruh resep,
2. normalize embedding,
3. masukkan ke FAISS,
4. simpan index dan metadata.

In [ ]:
corpus_texts = (
    df["passage_text"]
    .tolist()
)

corpus_embeddings = finetuned_model.encode(
    corpus_texts,
    batch_size=ENCODE_BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

print(
    "Embedding shape:",
    corpus_embeddings.shape
)

In [ ]:
embedding_dim = corpus_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dim
)

faiss_index.add(
    corpus_embeddings
)

print(
    "FAISS dimension:",
    embedding_dim
)

print(
    "Jumlah vector:",
    faiss_index.ntotal
)

In [ ]:
faiss.write_index(
    faiss_index,
    FAISS_PATH
)

metadata_cols = [
    "recipe_id",
    "Title",
    "ingredient_text",
    "Instructions",
    "rag_document",
    "retrieval_document",
]

if "Image_Name" in df.columns:
    metadata_cols.insert(
        2,
        "Image_Name"
    )

df[metadata_cols].to_csv(
    METADATA_PATH,
    index=False
)

print(
    "FAISS:",
    FAISS_PATH
)

print(
    "Metadata:",
    METADATA_PATH
)

# 11. User Query

In [ ]:
USER_QUERY = (
    "Saya punya ayam, kentang, "
    "bawang putih dan cabai. "
    "Bisa masak apa?"
)

print("USER:")
print(USER_QUERY)

# 12. Query Embedding

Query user diberi prefix `query:` lalu diubah menjadi embedding.

In [ ]:
query_text = (
    "query: "
    + USER_QUERY.strip()
)

query_embedding = finetuned_model.encode(
    [query_text],
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

print(
    "Query:",
    query_text
)

print(
    "Embedding shape:",
    query_embedding.shape
)

# 13. Similarity Search

In [ ]:
scores, indices = faiss_index.search(
    query_embedding,
    TOP_K
)

print("Scores:")
print(scores[0])

print("\nIndices:")
print(indices[0])

# 14. Top-K Context

In [ ]:
retrieved = (
    df.iloc[indices[0]]
      .copy()
)

retrieved["similarity_score"] = scores[0]

display(
    retrieved[
        [
            "recipe_id",
            "Title",
            "ingredient_text",
            "similarity_score"
        ]
    ]
)

In [ ]:
context_parts = []

for rank, (_, row) in enumerate(
    retrieved.iterrows(),
    start=1
):
    block = (
        f"[RECIPE {rank}]\n"
        f"Title: {row['Title']}\n"
        f"Ingredients: {row['ingredient_text']}\n"
        f"Instructions: {row['Instructions']}\n"
    )

    context_parts.append(block)

TOP_K_CONTEXT = "\n".join(
    context_parts
)

print(
    TOP_K_CONTEXT[:6000]
)

# 15. Prompt

In [ ]:
PROMPT = f"""
Anda adalah RacikAI, asisten resep berbasis Retrieval-Augmented Generation (RAG).

ATURAN:
1. Jawab terutama berdasarkan CONTEXT resep yang diberikan.
2. Jangan mengklaim resep berasal dari dataset jika tidak ada di CONTEXT.
3. Pilih resep yang paling sesuai dengan pertanyaan user.
4. Jika bahan user belum cukup, sebutkan bahan tambahan yang dibutuhkan.
5. Jawab dalam Bahasa Indonesia.
6. Buat langkah memasak ringkas dan mudah diikuti.
7. Di akhir, tuliskan "Sumber resep:" lalu judul resep yang digunakan.
8. Jika retrieval tidak relevan, katakan bahwa hasil retrieval belum cukup relevan.

CONTEXT:
{TOP_K_CONTEXT}

PERTANYAAN USER:
{USER_QUERY}

JAWABAN:
""".strip()

print(
    PROMPT[:8000]
)

# 16. LLM — Gemini API

Tambahkan `GEMINI_API_KEY` melalui **Kaggle Secrets**, jangan menulis API key langsung di notebook.

Model default notebook: `gemini-3.8-flash`.
Jika model tersebut belum tersedia untuk API key Anda, ubah variabel `GEMINI_MODEL`.

In [ ]:
from kaggle_secrets import UserSecretsClient
from google import genai

GEMINI_MODEL = "gemini-3.5-flash-lite"

user_secrets = UserSecretsClient()

try:
    gemini_api_key = (
        user_secrets.get_secret(
            "GEMINI_API_KEY"
        )
    )
except Exception:
    gemini_api_key = None

if gemini_api_key:
    client = genai.Client(
        api_key=gemini_api_key
    )

    print(
        "Gemini client siap."
    )
else:
    client = None

    print(
        "GEMINI_API_KEY belum ditemukan. "
        "Tambahkan melalui Kaggle Secrets."
    )

In [ ]:
if client is not None:
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=PROMPT,
    )

    FINAL_ANSWER = response.text
else:
    FINAL_ANSWER = (
        "LLM belum dijalankan karena "
        "GEMINI_API_KEY belum tersedia. "
        "Retrieval dan prompt sudah berhasil dibuat."
    )

print(FINAL_ANSWER)

# 17. Final — Fungsi RAG Lengkap

Pipeline online:

```text
User
 ↓
Query Embedding
 ↓
FAISS
 ↓
Top-K
 ↓
Context
 ↓
Prompt
 ↓
Gemini
 ↓
Final Answer
```

In [ ]:
def retrieve_recipes(
    user_query,
    top_k=5
):
    query = (
        "query: "
        + user_query.strip()
    )

    q_emb = finetuned_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(
        q_emb,
        top_k
    )

    results = (
        df.iloc[indices[0]]
          .copy()
    )

    results["similarity_score"] = (
        scores[0]
    )

    return results


def build_context(results):
    parts = []

    for rank, (_, row) in enumerate(
        results.iterrows(),
        start=1
    ):
        block = (
            f"[RECIPE {rank}]\n"
            f"Title: {row['Title']}\n"
            f"Ingredients: {row['ingredient_text']}\n"
            f"Instructions: {row['Instructions']}\n"
        )

        parts.append(block)

    return "\n".join(parts)


def build_rag_prompt(
    user_query,
    context
):
    prompt = f"""
Anda adalah RacikAI, asisten resep berbasis RAG.

Gunakan CONTEXT sebagai sumber utama.
Jawab dalam Bahasa Indonesia.
Jangan membuat sumber resep yang tidak ada di context.
Jika bahan user belum lengkap, sebutkan bahan tambahan.
Berikan langkah memasak yang ringkas.
Akhiri dengan judul sumber resep.

CONTEXT:
{context}

USER:
{user_query}

ANSWER:
""".strip()

    return prompt


def ask_recipe_rag(
    user_query,
    top_k=5
):
    results = retrieve_recipes(
        user_query,
        top_k=top_k
    )

    context = build_context(
        results
    )

    prompt = build_rag_prompt(
        user_query,
        context
    )

    if client is None:
        answer = (
            "Gemini belum aktif karena "
            "GEMINI_API_KEY belum tersedia."
        )
    else:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
        )

        answer = response.text

    return {
        "query": user_query,
        "retrieved": results[
            [
                "recipe_id",
                "Title",
                "ingredient_text",
                "similarity_score"
            ]
        ].reset_index(drop=True),
        "prompt": prompt,
        "answer": answer,
    }

In [ ]:
result = ask_recipe_rag(
    (
        "Saya ingin makanan dengan "
        "ikan paprika dan rasa pedas."
    ),
    top_k=5,
)

print("FINAL ANSWER:\n")
print(result["answer"])

print("\nTOP-K SOURCES:")
display(result["retrieved"])

# 18. Simpan Artefak untuk Backend / Flutter

Artefak:
- `/kaggle/working/recipe-e5-finetuned/`
- `/kaggle/working/recipe_faiss.index`
- `/kaggle/working/recipe_metadata.csv`

Backend FastAPI nantinya hanya perlu memuat model embedding, FAISS index, dan metadata. Flutter mengirim query ke backend.

In [ ]:
print("1. Model:", MODEL_DIR)
print("2. FAISS:", FAISS_PATH)
print("3. Metadata:", METADATA_PATH)

print("\n/kaggle/working:")
for p in Path("/kaggle/working").iterdir():
    print("-", p)

In [ ]:
"""Run in Kaggle after training to export the full model and matching index."""
import csv
import hashlib
import json
import tempfile
import zipfile
from pathlib import Path


def export_bundle(model, index_path, metadata_path, output_path):
    import faiss
    import numpy as np
    from sentence_transformers import SentenceTransformer
    from safetensors import safe_open

    index_path, metadata_path, output_path = map(Path, (index_path, metadata_path, output_path))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    index = faiss.read_index(str(index_path))
    with metadata_path.open(encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows or len(rows) != index.ntotal:
        raise ValueError("Jumlah baris metadata berbeda dengan index.")
    with tempfile.TemporaryDirectory(prefix="racikai-export-", dir=output_path.parent) as folder:
        root = Path(folder)
        model.save_pretrained(str(root / "model"), safe_serialization=True)
        with safe_open(root / "model" / "model.safetensors", framework="pt", device="cpu") as tensors:
            if not list(tensors.keys()):
                raise ValueError("Model kosong.")
        reloaded = SentenceTransformer(str(root / "model"), device="cpu", local_files_only=True)
        reloaded.max_seq_length = 384
        probes = [0, len(rows) // 2, len(rows) - 1]
        encoded = reloaded.encode(["passage: " + rows[i]["retrieval_document"] for i in probes],
                                  normalize_embeddings=True, convert_to_numpy=True)
        similarities = [float(np.dot(encoded[n], index.reconstruct(i))) for n, i in enumerate(probes)]
        if min(similarities) < 0.999:
            raise ValueError(f"Model dan index tidak cocok: {similarities}")
        files = {p.relative_to(root).as_posix(): p for p in (root / "model").rglob("*") if p.is_file()}
        files.update({"recipe_faiss.index": index_path, "recipe_metadata.csv": metadata_path})
        manifest = {"rows": len(rows), "dimension": index.d, "max_seq_length": 384,
                    "probe_cosine": similarities, "sha256": {}}
        for name, path in files.items():
            with path.open("rb") as stream:
                manifest["sha256"][name] = hashlib.file_digest(stream, "sha256").hexdigest()
        temporary_zip = root / "bundle.zip"
        with zipfile.ZipFile(temporary_zip, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
            for name, path in files.items():
                archive.write(path, name)
            archive.writestr("manifest.json", json.dumps(manifest, indent=2))
        with zipfile.ZipFile(temporary_zip) as archive:
            if archive.testzip() is not None:
                raise ValueError("Pemeriksaan ZIP gagal.")
        temporary_zip.replace(output_path)
    print(f"Bundle terverifikasi: {output_path} ({output_path.stat().st_size:,} byte)")
    return output_path


export_bundle(finetuned_model, FAISS_PATH, METADATA_PATH, "/kaggle/working/racikai-backend-bundle.zip")


In [ ]:
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/racikai-backend-bundle.zip"))


## 19. UI Sederhana dengan Gradio

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr


def gradio_recipe_ui(user_query, top_k):
    user_query = str(user_query).strip()

    if user_query == "":
        return (
            "Masukkan bahan atau pertanyaan terlebih dahulu.",
            pd.DataFrame()
        )

    # Retrieval dari fungsi yang sudah dibuat sebelumnya
    results = retrieve_recipes(
        user_query,
        top_k=int(top_k)
    )

    # Membuat context
    context = build_context(
        results
    )

    # Membuat prompt RAG
    prompt = build_rag_prompt(
        user_query,
        context
    )

    # Generate jawaban dengan Gemini
    if client is None:
        answer = (
            "Gemini belum aktif karena "
            "GEMINI_API_KEY belum tersedia."
        )
    else:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
        )

        answer = response.text

    # Tabel sumber hasil retrieval
    table = results[
        [
            "Title",
            "ingredient_text",
            "similarity_score"
        ]
    ].copy()

    table.columns = [
        "Judul Resep",
        "Bahan",
        "Similarity"
    ]

    table["Similarity"] = (
        table["Similarity"]
        .round(4)
    )

    return answer, table

In [ ]:
with gr.Blocks(
    title="RacikAI - Recipe RAG"
) as demo:

    gr.Markdown(
        """
        # 🍳 RacikAI

        Masukkan bahan yang kamu punya atau makanan
        yang ingin kamu buat.

        RacikAI akan mencari resep paling relevan
        menggunakan **FAISS + Fine-Tuned E5**
        kemudian menyusun jawaban dengan **Gemini**.
        """
    )

    query_input = gr.Textbox(
        label="Bahan / Pertanyaan",
        placeholder=(
            "Contoh: Saya punya ayam, kentang, "
            "bawang putih dan cabai. Bisa masak apa?"
        ),
        lines=3
    )

    top_k_input = gr.Slider(
        minimum=1,
        maximum=10,
        value=5,
        step=1,
        label="Jumlah resep yang diambil (Top-K)"
    )

    submit_button = gr.Button(
        "🔎 Cari Resep",
        variant="primary"
    )

    gr.Markdown("## 🤖 Jawaban RacikAI")

    answer_output = gr.Markdown()

    gr.Markdown("## 📚 Sumber Retrieval")

    sources_output = gr.Dataframe(
        headers=[
            "Judul Resep",
            "Bahan",
            "Similarity"
        ],
        interactive=False,
        wrap=True
    )

    submit_button.click(
        fn=gradio_recipe_ui,
        inputs=[
            query_input,
            top_k_input
        ],
        outputs=[
            answer_output,
            sources_output
        ]
    )

    query_input.submit(
        fn=gradio_recipe_ui,
        inputs=[
            query_input,
            top_k_input
        ],
        outputs=[
            answer_output,
            sources_output
        ]
    )

In [ ]:
demo.launch(
    share=True,
    debug=True,
    prevent_thread_lock=True
)

## 20. Deployment Hugging Face

In [ ]:
from pathlib import Path
import shutil
import os

SPACE_DIR = Path(
    "/kaggle/working/racikai_hf_space"
)

SPACE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Folder Space:",
    SPACE_DIR
)

In [ ]:
MODEL_SOURCE = Path(
    "/kaggle/working/recipe-e5-finetuned"
)

FAISS_SOURCE = Path(
    "/kaggle/working/recipe_faiss.index"
)

METADATA_SOURCE = Path(
    "/kaggle/working/recipe_metadata.csv"
)


MODEL_DEST = (
    SPACE_DIR /
    "recipe-e5-finetuned"
)

FAISS_DEST = (
    SPACE_DIR /
    "recipe_faiss.index"
)

METADATA_DEST = (
    SPACE_DIR /
    "recipe_metadata.csv"
)


if MODEL_DEST.exists():
    shutil.rmtree(
        MODEL_DEST
    )

shutil.copytree(
    MODEL_SOURCE,
    MODEL_DEST
)

shutil.copy2(
    FAISS_SOURCE,
    FAISS_DEST
)

shutil.copy2(
    METADATA_SOURCE,
    METADATA_DEST
)


print("Artefak berhasil dicopy.")

In [ ]:
app_code = r'''
import os
from pathlib import Path

import faiss
import gradio as gr
import pandas as pd

from google import genai
from sentence_transformers import SentenceTransformer


# =========================================
# PATH
# =========================================

BASE_DIR = Path(
    __file__
).resolve().parent

MODEL_DIR = (
    BASE_DIR /
    "recipe-e5-finetuned"
)

FAISS_PATH = (
    BASE_DIR /
    "recipe_faiss.index"
)

METADATA_PATH = (
    BASE_DIR /
    "recipe_metadata.csv"
)


# =========================================
# GEMINI
# =========================================

GEMINI_MODEL = os.getenv(
    "GEMINI_MODEL",
    "gemini-3.5-flash-lite"
)


# =========================================
# CEK FILE
# =========================================

def check_artifacts():

    missing = []

    for path in [
        MODEL_DIR,
        FAISS_PATH,
        METADATA_PATH
    ]:

        if not path.exists():
            missing.append(
                path.name
            )

    if missing:

        raise FileNotFoundError(
            "Artefak belum lengkap: "
            + ", ".join(missing)
        )


check_artifacts()


# =========================================
# LOAD MODEL
# =========================================

print(
    "Memuat model embedding..."
)

embedding_model = SentenceTransformer(
    str(MODEL_DIR),
    device="cpu"
)

embedding_model.max_seq_length = 384


# =========================================
# LOAD FAISS
# =========================================

print(
    "Memuat FAISS index..."
)

faiss_index = faiss.read_index(
    str(FAISS_PATH)
)


# =========================================
# LOAD METADATA
# =========================================

print(
    "Memuat metadata..."
)

metadata = pd.read_csv(
    METADATA_PATH
)


# =========================================
# GEMINI CLIENT
# =========================================

api_key = os.getenv(
    "GEMINI_API_KEY"
)

if api_key:

    client = genai.Client(
        api_key=api_key
    )

else:

    client = None


# =========================================
# RETRIEVAL
# =========================================

def retrieve_recipes(
    user_query,
    top_k=5
):

    query = (
        "query: "
        + user_query.strip()
    )

    q_emb = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(
        "float32"
    )

    scores, indices = (
        faiss_index.search(
            q_emb,
            int(top_k)
        )
    )

    valid = (
        indices[0] >= 0
    )

    idx = (
        indices[0][valid]
    )

    scr = (
        scores[0][valid]
    )

    results = (
        metadata
        .iloc[idx]
        .copy()
    )

    results[
        "similarity_score"
    ] = scr

    return results


# =========================================
# CONTEXT
# =========================================

def build_context(
    results
):

    parts = []

    for rank, (_, row) in enumerate(
        results.iterrows(),
        start=1
    ):

        block = (
            f"[RECIPE {rank}]\n"
            f"Title: {row['Title']}\n"
            f"Ingredients: "
            f"{row['ingredient_text']}\n"
            f"Instructions: "
            f"{row['Instructions']}\n"
        )

        parts.append(
            block
        )

    return "\n".join(
        parts
    )


# =========================================
# PROMPT
# =========================================

def build_rag_prompt(
    user_query,
    context
):

    return f"""
Anda adalah RacikAI,
asisten resep berbasis
Retrieval-Augmented Generation (RAG).

ATURAN:

1. Gunakan CONTEXT sebagai sumber utama.

2. Jawab menggunakan Bahasa Indonesia.

3. Jangan membuat sumber resep yang
tidak tersedia pada CONTEXT.

4. Jika bahan user belum lengkap,
sebutkan bahan tambahan yang dibutuhkan.

5. Berikan langkah memasak yang
ringkas dan mudah diikuti.

6. Akhiri jawaban dengan bagian:
"Sumber resep:"
dan tuliskan judul resep yang digunakan.

7. Jika hasil retrieval tidak cukup
relevan, sampaikan dengan jelas.


CONTEXT:

{context}


PERTANYAAN USER:

{user_query}


JAWABAN:
""".strip()


# =========================================
# RAG
# =========================================

def ask_recipe(
    user_query,
    top_k
):

    user_query = (
        user_query or ""
    ).strip()

    if not user_query:

        return (
            "Masukkan pertanyaan "
            "atau bahan terlebih dahulu.",
            pd.DataFrame()
        )


    # Retrieval
    results = retrieve_recipes(
        user_query,
        top_k
    )


    # Context
    context = build_context(
        results
    )


    # Prompt
    prompt = build_rag_prompt(
        user_query,
        context
    )


    # Gemini
    if client is None:

        answer = (
            "GEMINI_API_KEY belum "
            "diatur pada Hugging Face "
            "Space Secrets.\n\n"
            "Retrieval FAISS sudah "
            "berjalan tetapi Gemini "
            "belum dapat menghasilkan "
            "jawaban."
        )

    else:

        try:

            response = (
                client.models
                .generate_content(
                    model=GEMINI_MODEL,
                    contents=prompt
                )
            )

            answer = (
                response.text
                or
                "Gemini tidak "
                "mengembalikan jawaban."
            )

        except Exception as e:

            answer = (
                "Terjadi error saat "
                "menghubungi Gemini:\n\n"
                + str(e)
            )


    # Table
    table = results[
        [
            "Title",
            "ingredient_text",
            "similarity_score"
        ]
    ].copy()


    table.columns = [
        "Judul Resep",
        "Bahan",
        "Similarity"
    ]


    table[
        "Similarity"
    ] = (
        table[
            "Similarity"
        ].round(4)
    )


    return (
        answer,
        table
    )


# =========================================
# GRADIO UI
# =========================================

with gr.Blocks(
    title="RacikAI"
) as demo:

    gr.Markdown(
        """
        # 🍳 RacikAI

        ### AI Recipe Assistant berbasis RAG

        Masukkan bahan yang kamu punya
        atau makanan yang ingin dibuat.

        Sistem akan:

        **Query → Embedding → FAISS Retrieval
        → Context → Gemini → Jawaban**
        """
    )


    query_input = gr.Textbox(
        label="🥕 Bahan / Pertanyaan",
        placeholder=(
            "Contoh: Saya punya ayam, "
            "kentang, bawang putih dan "
            "cabai. Bisa masak apa?"
        ),
        lines=3
    )


    top_k_input = gr.Slider(
        minimum=1,
        maximum=10,
        value=5,
        step=1,
        label=(
            "Jumlah resep yang "
            "diambil (Top-K)"
        )
    )


    submit_button = gr.Button(
        "🔎 Cari Resep",
        variant="primary"
    )


    gr.Markdown(
        "## 🤖 Jawaban RacikAI"
    )


    answer_output = gr.Markdown()


    gr.Markdown(
        "## 📚 Sumber Retrieval"
    )


    sources_output = gr.Dataframe(
        headers=[
            "Judul Resep",
            "Bahan",
            "Similarity"
        ],
        interactive=False,
        wrap=True
    )


    submit_button.click(
        fn=ask_recipe,
        inputs=[
            query_input,
            top_k_input
        ],
        outputs=[
            answer_output,
            sources_output
        ]
    )


    query_input.submit(
        fn=ask_recipe,
        inputs=[
            query_input,
            top_k_input
        ],
        outputs=[
            answer_output,
            sources_output
        ]
    )


# =========================================
# RUN
# =========================================

if __name__ == "__main__":

    demo.launch()
'''

with open(
    SPACE_DIR / "app.py",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        app_code
    )


print(
    "app.py berhasil dibuat."
)

In [ ]:
requirements = """
gradio>=5.0,<7.0
sentence-transformers>=3.0,<6.0
faiss-cpu>=1.8,<2.0
pandas>=2.0,<3.0
numpy>=1.26,<3.0
google-genai>=1.0,<2.0
torch>=2.1
""".strip()


with open(
    SPACE_DIR / "requirements.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        requirements
    )


print(
    "requirements.txt berhasil dibuat."
)

In [ ]:
readme = """---
title: RacikAI
emoji: 🍳
colorFrom: green
colorTo: yellow
sdk: gradio
app_file: app.py
pinned: false
---

# RacikAI

RacikAI adalah AI Recipe Assistant
berbasis Retrieval-Augmented Generation.

Pipeline:

User Query  
↓  
Fine-Tuned Multilingual E5  
↓  
FAISS Retrieval  
↓  
Top-K Recipe Context  
↓  
Gemini  
↓  
Jawaban Resep
"""


with open(
    SPACE_DIR / "README.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        readme
    )


print(
    "README.md berhasil dibuat."
)

In [ ]:
import shutil

ZIP_PATH = (
    "/kaggle/working/"
    "racikai_hf_space"
)

shutil.make_archive(
    ZIP_PATH,
    "zip",
    SPACE_DIR
)

print(
    "ZIP selesai dibuat:"
)

print(
    ZIP_PATH + ".zip"
)